## Iteration 1: Dataset Loading

### Objective
In this step, we will load the **Flowers Dataset**, a collection of flower images provided by TensorFlow. The dataset contains five categories: daisies, dandelions, roses, sunflowers, and tulips. The goal is to extract the dataset, load it into TensorFlow, and split it into training and validation subsets.

### Tasks
1. Download the dataset from TensorFlow’s public URL.
2. Extract the dataset into a structured folder format.


In [ ]:
import tensorflow as tf
import os

# Dataset URL
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

# Current directory
current_dir = os.getcwd()

# Download the dataset to the current directory
dataset_path = tf.keras.utils.get_file(
    fname="flower_photos.tgz",
    origin=url,
    extract=True,
    cache_dir=current_dir,
    cache_subdir=""
)

# Extracted dataset directory
dataset_dir = os.path.join(os.path.dirname(dataset_path), 'flower_photos')
print(f"Dataset extracted at: {dataset_dir}")

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Base dataset directory (adapted to your code)
dataset_dir = "./flower_photos_extracted"  # Ajusta esta ruta según tu contexto si es necesario

# Dynamically create train and val directories
base_split_dir = os.path.join(os.path.dirname(dataset_dir), "flower_photos_split")
train_dir = os.path.join(base_split_dir, "train")
val_dir = os.path.join(base_split_dir, "val")

# Create folders for train and val
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Split the images for each class
for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):  # Ensure it's a class folder
        # Get all images of the class
        images = os.listdir(class_path)

        # Split images into training and validation (80/20)
        train_images, val_images = train_test_split(images, test_size=0.2, random_state=42)

        # Create folders for the class in train and val
        os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
        os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

        # Move the images to the corresponding folders
        for img in train_images:
            shutil.move(os.path.join(class_path, img), os.path.join(train_dir, class_name, img))
        for img in val_images:
            shutil.move(os.path.join(class_path, img), os.path.join(val_dir, class_name, img))

print(f"Train directory: {train_dir}")
print(f"Validation directory: {val_dir}")

## Iteration 2: Data Preprocessing

### Objective
Preprocess the images to make them suitable for the MobileNetV2 model. This includes normalizing pixel values and applying data augmentation to enhance model generalization.

### Tasks
1. Normalize the pixel values of the images to a range of 0 to 1 using `Rescaling`.
2. Apply data augmentation techniques:
    - Random flips
    - Rotations
    - Zoom
3. Optimize the dataset pipeline with `tf.data`:
    - Shuffle the dataset.
    - Prefetch batches to improve training performance.
4. Confirm that the preprocessing steps are applied correctly by visualizing a batch of augmented images.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation for the training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255) # No augmentation, just rescaling

# Dataset paths
train_dir = "flower_photos_split/train"
val_dir = "flower_photos_split/val"

# Create the data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
import matplotlib.pyplot as plt

# Get a batch of images
images, labels = next(train_generator)

# Display some images
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')
    plt.title(f"Class: {labels[i].argmax()}")
plt.show()

## Iteration 3: Load a Pre-trained Model

### Objective
Leverage the pre-trained **MobileNetV2** model for transfer learning. Configure the model to adapt it for the five flower categories.

### Tasks
1. Load the MobileNetV2 model with weights pre-trained on the ImageNet dataset.
2. Freeze the base model layers to retain the pre-trained features.
3. Add custom classification layers:
    - A global average pooling layer.
    - A dense layer with ReLU activation and 1024 neurons.
    - An output layer with softmax activation for five classes.
4. Compile the model with a categorical cross-entropy loss and an appropriate optimizer, use a moderate learning rate.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Load pre-trained MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base layers to preserve pre-trained weights

# Add custom layers to the model

# Final model

# Compile the model

# Model summary
model.summary()


## Iteration 4: Fine-tune the Pre-trained Model

### Objective
Train and fine-tune the model to adapt the pre-trained features to the flower dataset.

### Tasks
1. Train the custom classification layers while keeping the base model frozen:
    - Set **`epochs`** to `5` for a short training duration.
    - Set **`steps_per_epoch`** to `50` to adjust for the dataset size.
    - Set **`validation_steps`** to `10` to evaluate on a subset of the validation data.
    - Monitor loss and accuracy on both training and validation datasets.
2. Visualize the training process:
    - Plot the training and validation accuracy.
    - Plot the training and validation loss.

In [ ]:
from transformers import TFDistilBertForSequenceClassification
import tensorflow as tf

# Fit the nmodel

## Iteration 5: Model Evaluation

### Objective
Evaluate the model’s performance on the validation dataset using various metrics and visualizations.

### Tasks
1. Compute overall accuracy on the validation set.
2. Generate a confusion matrix to visualize class-wise performance.
3. Calculate precision, recall, and F1-score for each class.

In [ ]:
import matplotlib.pyplot as plt

# Graficar la precisión


# Graficar la pérdida


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Generate a confusion matrix to visualize class-wise performance
# Get predictions and true labels


# Confusion matrix


# Visualize the confusion matrix


# 3. Calculate precision, recall, and F1-score for each class


## Iteration 6: Save and Load the Model

### Objective
Save the fine-tuned model to ensure it can be reused without retraining. Demonstrate how to reload the model for predictions.

### Tasks
1. Save the model using TensorFlow’s `model.save` method.
2. Load the saved model to verify that it works correctly.
3. Make predictions on new data using the loaded model.

In [ ]:
# Save the model
model.save('flower_classifier_mobilenetv2.h5')

# Load the model
from tensorflow.keras.models import load_model
loaded_model = load_model('flower_classifier_mobilenetv2.h5')

# Evaluate the loaded model
loss, accuracy = loaded_model.evaluate(val_generator)
print(f"Loss: {loss}, Accuracy: {accuracy}")

## Iteration 7: Custom Predictions

### Objective
Test the model on custom images to validate its generalization on unseen data.

### Tasks
1. Provide a set of custom flower images.
2. Preprocess the images to match the input size and format of the model.
3. Use the model to predict the class for each image.
4. Display the images with their predicted labels and confidence scores.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
import os

# Make sure to load your model before running the code
# model = ... (load your model beforehand)

# Define the image paths
image_folder = './test_images'  # Change this if the images are in a different folder
image_names = [f"Image_{i}.jpg" for i in range(1, 21)]  # Image names (Image_1.jpg to Image_20.jpg)

# Get the class labels
class_labels = list(train_generator.class_indices.keys())

# Iterate over the images
for img_name in image_names:
    img_path = os.path.join(image_folder, img_name)

    try:
        # Load and preprocess the image
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)  # Add dimension for the batch
        img_array /= 255.0  # Normalization

        # Make the prediction
        predictions = model.predict(img_array)
        class_index = np.argmax(predictions)

        # Get the predicted class name
        predicted_class = class_labels[class_index]

        print(f"Image: {img_name} - Predicted class: {predicted_class}")
    except Exception as e:
        print(f"Error processing the image {img_name}: {e}")
